In [ ]:
import numpy as np
import nibabel as nib
from dipy.io.image import load_nifti
from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.dti import TensorModel
from dipy.tracking import utils
from dipy.tracking.local_tracking import LocalTracking
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking.streamline import Streamlines
from dipy.data import default_sphere
from dipy.direction import DeterministicMaximumDirectionGetter
from dipy.tracking.utils import connectivity_matrix

# Load data
data, affine = load_nifti('dwi.nii.gz')
bvals, bvecs = read_bvals_bvecs('bvals', 'bvecs')
gtab = gradient_table(bvals, bvecs)

# Fit tensor model
tenmodel = TensorModel(gtab)
tenfit = tenmodel.fit(data)

# Compute FA
FA = tenfit.fa

# Create stopping criterion
stopping_criterion = ThresholdStoppingCriterion(FA, 0.2)

# Define seeds
seed_mask = FA > 0.2
seeds = utils.seeds_from_mask(seed_mask, affine, density=1)

# Create direction getter
from dipy.reconst.dti import quantize_evecs
peak_indices = quantize_evecs(tenfit.evecs, default_sphere.vertices)
direction_getter = DeterministicMaximumDirectionGetter.from_index(peak_indices, max_angle=30., sphere=default_sphere)

# Perform tractography
streamlines_generator = LocalTracking(direction_getter, stopping_criterion, seeds, affine, step_size=0.5)
streamlines = Streamlines(streamlines_generator)

# Load atlas
atlas_data, atlas_affine = load_nifti('atlas.nii.gz')

# Compute connectivity matrix
M, grouping = connectivity_matrix(streamlines, atlas_affine, atlas_data, return_mapping=True, mapping_as_streamlines=False)


In [ ]:
import slicer
import os

# Define file paths
dwi_path = '/path/to/dwi.nrrd'
bvals_path = '/path/to/bvals'
bvecs_path = '/path/to/bvecs'
atlas_path = '/path/to/atlas.nii.gz'
t1_path = '/path/to/T1.nii.gz'

# Load DWI data
[dwi_node, dwi_loaded] = slicer.util.loadVolume(dwi_path, returnNode=True)

# Create diffusion tensor estimation module
dti_module = slicer.modules.diffusiontensorestimation

# Set parameters
parameters = {
    'inputVolume': dwi_node.GetID(),
    'outputTensorVolume': 'DTI',
    'bValues': bvals_path,
    'bVectors': bvecs_path
}

# Run the module
slicer.cli.runSync(dti_module, None, parameters)

# Retrieve the output tensor volume
dti_node = slicer.util.getNode('DTI')

# Create diffusion tensor scalar measurements module
scalar_module = slicer.modules.diffusiontensorscalarmeasurements

# Set parameters
parameters = {
    'inputVolume': dti_node.GetID(),
    'outputFA': 'FA',
    'scalarMeasurement': 'FractionalAnisotropy'
}

# Run the module
slicer.cli.runSync(scalar_module, None, parameters)

# Retrieve the FA volume
fa_node = slicer.util.getNode('FA')

# Create tractography seeding module
tract_module = slicer.modules.tractographylabelmapseeding

# Set parameters
parameters = {
    'inputVolume': dti_node.GetID(),
    'seedLabel': fa_node.GetID(),
    'outputFiberBundle': 'Tractography',
    'stoppingValue': 0.2,
    'minimumPathLength': 10,
    'maximumPathLength': 200,
    'integrationStepLength': 0.5,
    'seedingRegion': 'SeedRegion'
}

# Run the module
slicer.cli.runSync(tract_module, None, parameters)

# Retrieve the fiber bundle
tract_node = slicer.util.getNode('Tractography')

# Load T1-weighted image
[t1_node, t1_loaded] = slicer.util.loadVolume(t1_path, returnNode=True)

# Load atlas
[atlas_node, atlas_loaded] = slicer.util.loadVolume(atlas_path, returnNode=True)

# Perform registration (e.g., using BRAINSFit)
registration_module = slicer.modules.brainsfit

parameters = {
    'fixedVolume': t1_node.GetID(),
    'movingVolume': atlas_node.GetID(),
    'linearTransform': 'AtlasToT1Transform'
}

slicer.cli.runSync(registration_module, None, parameters)

# Apply the transform to the atlas
transform_node = slicer.util.getNode('AtlasToT1Transform')
atlas_node.SetAndObserveTransformNodeID(transform_node.GetID())
slicer.vtkSlicerTransformLogic().hardenTransform(atlas_node)

# Create fiber tract measurements module
connectivity_module = slicer.modules.fibertractmeasurements

# Set parameters
parameters = {
    'inputFiberBundle': tract_node.GetID(),
    'inputLabelVolume': atlas_node.GetID(),
    'outputTable': 'ConnectivityMatrix',
    'measurements': ['MeanFA', 'FiberLengthMean']
}

# Run the module
slicer.cli.runSync(connectivity_module, None, parameters)

# Retrieve the output table
connectivity_table = slicer.util.getNode('ConnectivityMatrix')
